In [ ]:
#Install dependencies
!pip install wfdb scipy imbalanced-learn -q

import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import kagglehub

# 1. Load Data
path = kagglehub.dataset_download("shayanfazeli/heartbeat")
train_df = pd.read_csv(os.path.join(path, 'mitbih_train.csv'), header=None)
test_df  = pd.read_csv(os.path.join(path, 'mitbih_test.csv'),  header=None)

X_train_raw = train_df.iloc[:, :-1].values
y_train_raw = train_df.iloc[:, -1].values.astype(int)
X_test_raw  = test_df.iloc[:, :-1].values
y_test_raw  = test_df.iloc[:, -1].values.astype(int)

class_names = ['N', 'S', 'V', 'F', 'Q']
num_classes = 5

print('Original train class distribution:')
unique, counts = np.unique(y_train_raw, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {class_names[u]}: {c}')

# 2. Preprocessing
from scipy.signal import butter, filtfilt

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=125.0, order=4):
    nyq  = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

def preprocess(X):
    X_f = np.array([bandpass_filter(x) for x in X])
    X_min = X_f.min(axis=1, keepdims=True)
    X_max = X_f.max(axis=1, keepdims=True)
    return (X_f - X_min) / (X_max - X_min + 1e-8)

print('Preprocessing...')
X_train_proc = preprocess(X_train_raw)
X_test_proc  = preprocess(X_test_raw)

# 3. SMOTE
print('Applying SMOTE...')
smote = SMOTE(
    sampling_strategy={
        1: 5000,
        2: 6000,
        3: 3000,
        4: 7000,
    },
    random_state=42,
    k_neighbors=5
)
X_train_sm, y_train_sm = smote.fit_resample(X_train_proc, y_train_raw)

print('Post-SMOTE class distribution:')
unique, counts = np.unique(y_train_sm, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {class_names[u]}: {c}')

#  4. Train-Test Split + Reshape
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_sm, y_train_sm,
    test_size=0.1, random_state=42, stratify=y_train_sm
)

X_tr_cnn  = X_tr.reshape(X_tr.shape[0],   X_tr.shape[1],  1)
X_val_cnn = X_val.reshape(X_val.shape[0],  X_val.shape[1], 1)
X_te_cnn  = X_test_proc.reshape(X_test_proc.shape[0], X_test_proc.shape[1], 1)

print('Train shape:', X_tr_cnn.shape)

# 5. Focal Loss penalizes easy examples, focuses on hard minority classes
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -y_true_oh * tf.math.log(y_pred)
        weight    = alpha * y_true_oh * tf.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss_fn

#  6.CNN Architecture
def build_cnn(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(64, kernel_size=5, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    # Block 2
    x = layers.Conv1D(128, kernel_size=5, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    # Block 3
    x = layers.Conv1D(256, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    # Block 4
    x = layers.Conv1D(128, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)

model = build_cnn((187, 1), num_classes)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)
model.summary()

#  7. Train
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10, restore_best_weights=True
)
lr_reduce = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
)

history = model.fit(
    X_tr_cnn, y_tr,
    epochs=80,
    batch_size=128,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stop, lr_reduce],
    verbose=1
)

# 8. Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Focal Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

# 9. Evaluation
y_prob = model.predict(X_te_cnn)
y_pred = np.argmax(y_prob, axis=1)

print(f'Test Accuracy : {accuracy_score(y_test_raw, y_pred):.4f}')
print(f'Macro F1      : {f1_score(y_test_raw, y_pred, average="macro"):.4f}')
print(f'Weighted F1   : {f1_score(y_test_raw, y_pred, average="weighted"):.4f}')
print(f'AUC-ROC (OvR) : {roc_auc_score(y_test_raw, y_prob, multi_class="ovr", average="macro"):.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test_raw, y_pred, target_names=class_names))

# 10. Confusion Matrix
cm = confusion_matrix(y_test_raw, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Improved 1D CNN')
plt.tight_layout(); plt.show()

In [ ]:
!pip install pdfplumber opencv-python-headless scipy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import cv2
from scipy.signal import butter, filtfilt, resample, find_peaks
from PIL import Image
import io

PDF_PATH = '/content/TEST3.pdf'

#  1. Extract Page Image
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]
    img  = page.to_image(resolution=300)
    buf  = io.BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    page_img = np.array(Image.open(buf).convert('RGB'))

print('Page image shape:', page_img.shape)

# 2. HSV Mask for Orange Trace
hsv  = cv2.cvtColor(page_img, cv2.COLOR_RGB2HSV)
mask = cv2.inRange(hsv, np.array([5, 100, 100]), np.array([25, 255, 255]))
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

# 3. Detect Exactly 3 ECG Strips (skip calibration pulse)
H, W        = mask.shape
row_sums    = mask.sum(axis=1)
active_rows = np.where(row_sums > W * 0.02)[0]

strip_bounds = []
if len(active_rows) > 0:
    gaps   = np.where(np.diff(active_rows) > 30)[0]
    starts = np.concatenate([[active_rows[0]], active_rows[gaps + 1]])
    ends   = np.concatenate([active_rows[gaps], [active_rows[-1]]])
    for s, e in zip(starts, ends):
        if e - s > 100:   # skip calibration pulse
            strip_bounds.append((s, e))

print(f'Detected {len(strip_bounds)} ECG strips: {strip_bounds}')
assert len(strip_bounds) == 3, f'Expected 3 strips, got {len(strip_bounds)} — tune threshold'

# 4. Extract + Resample Each Strip to 1250 Samples (10s @ 125Hz)
def extract_strip_signal(mask_strip):
    signal = []
    for col in range(mask_strip.shape[1]):
        rows = np.where(mask_strip[:, col] > 0)[0]
        signal.append(np.mean(rows) if len(rows) > 0 else np.nan)
    signal = np.array(signal)
    nans = np.isnan(signal)
    if nans.any():
        x = np.arange(len(signal))
        signal[nans] = np.interp(x[nans], x[~nans], signal[~nans])
    return -signal  # invert: high pixel row = low voltage

TARGET_FS         = 125
SAMPLES_PER_STRIP = TARGET_FS * 10   # 1250 samples per 10s strip

all_strips = []
for i, (r0, r1) in enumerate(strip_bounds):
    sig = extract_strip_signal(mask[r0:r1, :])
    sig_resampled = resample(sig, SAMPLES_PER_STRIP)
    all_strips.append(sig_resampled)
    plt.figure(figsize=(14, 2))
    plt.plot(sig_resampled, color='darkorange', linewidth=0.8)
    plt.title(f'Strip {i+1} — Resampled to {SAMPLES_PER_STRIP} samples')
    plt.xlabel('Samples @125Hz'); plt.tight_layout(); plt.show()

ecg_125hz = np.concatenate(all_strips)
print(f'Total signal: {len(ecg_125hz)} samples @ {TARGET_FS} Hz (expected {30*TARGET_FS})')

# 5. Bandpass Filter + Normalize
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=125.0, order=4):
    nyq  = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

ecg_filtered = bandpass_filter(ecg_125hz, fs=TARGET_FS)
ecg_norm     = (ecg_filtered - ecg_filtered.min()) / (ecg_filtered.max() - ecg_filtered.min() + 1e-8)

plt.figure(figsize=(16, 3))
plt.plot(ecg_norm, color='steelblue', linewidth=0.7)
plt.title('Reconstructed ECG — Filtered & Normalized (125 Hz)')
plt.xlabel('Samples'); plt.ylabel('Amplitude')
plt.tight_layout(); plt.show()

# 6. R-Peak Detection
r_peaks, _ = find_peaks(ecg_norm, height=0.45, distance=45, prominence=0.2)
print(f'R-peaks detected : {len(r_peaks)}')
print(f'Expected (~108 bpm × 30s) : ~{int(108*30/60)} beats')

plt.figure(figsize=(16, 3))
plt.plot(ecg_norm, color='steelblue', linewidth=0.7, label='ECG')
plt.plot(r_peaks, ecg_norm[r_peaks], 'rv', markersize=8, label='R-peaks')
plt.title('R-Peak Detection'); plt.xlabel('Samples')
plt.legend(); plt.tight_layout(); plt.show()

#  7. Segment Heartbeats
SEGMENT_LEN = 187
half        = SEGMENT_LEN // 2
segments, valid_peaks = [], []

for peak in r_peaks:
    start = peak - half
    end   = start + SEGMENT_LEN
    if start >= 0 and end <= len(ecg_norm):
        seg = ecg_norm[start:end]
        seg = (seg - seg.min()) / (seg.max() - seg.min() + 1e-8)
        segments.append(seg)
        valid_peaks.append(peak)

segments    = np.array(segments)
valid_peaks = np.array(valid_peaks)
print(f'Valid segments: {len(segments)}')



In [ ]:

# CELL 2 — CLASSIFICATION

from scipy.signal import resample as sp_resample
import numpy as np
import matplotlib.pyplot as plt
import os

SAVE_DIR  = '/content/ecg_images/'
os.makedirs(SAVE_DIR, exist_ok=True)

class_names  = ['N (Normal)', 'S (SVEB)', 'V (VEB)', 'F (Fusion)', 'Q (Unknown)']
class_colors = {0: 'green', 1: 'royalblue', 2: 'red', 3: 'orange', 4: 'purple'}

SEG_LEN   = 187
TARGET_FS = 125

#  Calculate RR interval from detected peaks
mean_rr  = int(np.mean(np.diff(r_peaks)))
WIN_SIZE = int(mean_rr * 0.80)   # 80% of RR — one beat only, no overlap
PRE      = WIN_SIZE // 3          # peak sits at 1/3 from left
POST     = WIN_SIZE - PRE - 1

print("=" * 55)
print("   GALAXY WATCH ECG — CLASSIFICATION")
print("=" * 55)
print(f"  Detected peaks   : {len(r_peaks)}")
print(f"  Mean RR interval : {mean_rr} samples ({TARGET_FS*60/mean_rr:.1f} bpm)")
print(f"  Window size      : {WIN_SIZE} samples (80% of RR)")
print(f"  PRE / POST       : {PRE} / {POST}")
print(f"  Resampled to     : {SEG_LEN} samples for model")


#  Segment heartbeats
segments    = []
valid_peaks = []

for peak in r_peaks:
    start = peak - PRE
    end   = peak + POST + 1
    if start < 0 or end > len(ecg_norm):
        continue
    seg = ecg_norm[start:end].copy()
    if len(seg) < 10:
        continue
    # Resample short window to 187 to match model input
    seg_resampled = sp_resample(seg, SEG_LEN)
    seg_min = seg_resampled.min()
    seg_max = seg_resampled.max()
    if seg_max - seg_min > 1e-8:
        seg_resampled = (seg_resampled - seg_min) / (seg_max - seg_min)
    segments.append(seg_resampled)
    valid_peaks.append(peak)

segments    = np.array(segments)
valid_peaks = np.array(valid_peaks)
print(f"\n  Valid segments   : {len(segments)}")


# CNN Inference
print("  Running CNN inference...")
X_input = segments.reshape(len(segments), SEG_LEN, 1)
y_prob  = model.predict(X_input, verbose=0)
y_pred  = np.argmax(y_prob, axis=1)
conf    = np.max(y_prob, axis=1)
print(f"  ✓ {len(y_pred)} beats classified")

unique, counts = np.unique(y_pred, return_counts=True)
dominant_class = unique[np.argmax(counts)]

diagnosis_map = {
    0: "Normal Sinus Rhythm",
    1: "Supraventricular Ectopic Beats Detected",
    2: "Ventricular Ectopic Beats Detected",
    3: "Fusion Beats Detected",
    4: "Unknown / Paced Beats Detected"
}
diagnosis     = diagnosis_map[dominant_class]
dominant_bpm  = TARGET_FS * 60 / mean_rr


#  Per-beat results table
print("\n" + "=" * 45)
print("         PER-BEAT PREDICTIONS")
print("=" * 45)
print(f"{'Beat':>6}  {'Class':<14}  {'Confidence':>10}")
print("-" * 38)
for i, (pred, c) in enumerate(zip(y_pred, conf)):
    flag = "  ⚠" if pred in [2, 3] and c > 0.75 else ""
    print(f"{i+1:>6}  {class_names[pred]:<14}  {c*100:>9.1f}%{flag}")

print("\n" + "=" * 45)
print("         OVERALL SUMMARY")
print("=" * 45)
for u, c_count in zip(unique, counts):
    print(f"  {class_names[u]:<14}: {c_count:>3} beats  ({c_count/len(y_pred)*100:.1f}%)")
print(f"\n  Total beats      : {len(y_pred)}")
print(f"  Detected BPM     : {dominant_bpm:.1f}")
print(f"  Mean confidence  : {conf.mean()*100:.1f}%")
print(f"  High conf >80%   : {(conf > 0.8).sum()} beats")
print(f"  Low  conf <50%   : {(conf < 0.5).sum()} beats")
print(f"\n  Diagnosis        : {diagnosis}")
print("=" * 45)


#  FIGURE 1: Annotated ECG
plotted = set()
fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(ecg_norm, color='lightgray', linewidth=0.7, zorder=1)
for peak, pred in zip(valid_peaks, y_pred):
    lbl = class_names[pred] if pred not in plotted else None
    ax.axvline(x=peak, color=class_colors[pred],
               alpha=0.85, linewidth=1.3, label=lbl, zorder=2)
    plotted.add(pred)
ax.set_title(
    f'Galaxy Watch ECG — CNN Per-Beat Classification\n'
    f'BPM: {dominant_bpm:.1f}  |  '
    f'Beats: {len(y_pred)}  |  '
    f'Diagnosis: {diagnosis}',
    fontsize=11
)
ax.set_xlabel('Samples (@125 Hz)'); ax.set_ylabel('Amplitude')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.2); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ecg_classification.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  ✓ Saved: ecg_classification.png")


# FIGURE 2: First 10 beat windows
n_show = min(10, len(segments))
fig, axes = plt.subplots(2, 5, figsize=(18, 5))
axes = axes.flatten()
for i in range(n_show):
    col = class_colors[y_pred[i]]
    axes[i].plot(segments[i], color=col, linewidth=1.2)
    axes[i].axvline(x=SEG_LEN//3, color='gray',
                    linestyle='--', alpha=0.5, linewidth=0.8)
    axes[i].set_title(
        f'Beat {i+1}\n{class_names[y_pred[i]]}\n{conf[i]*100:.1f}%',
        fontsize=8, color=col
    )
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xticks([]); axes[i].set_yticks([])
plt.suptitle('First 10 Heartbeat Segments', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ecg_beat_windows.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  ✓ Saved: ecg_beat_windows.png")


#  FIGURE 3: Confidence distribution + class pie
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(conf*100, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=conf.mean()*100, color='red', linestyle='--',
                linewidth=1.5, label=f'Mean: {conf.mean()*100:.1f}%')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].set_xlabel('Confidence (%)'); axes[0].set_ylabel('Number of beats')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

class_counts = [int((y_pred == c).sum()) for c in range(5)]
labels_pie   = [f"{class_names[c]}\n({class_counts[c]})"
                for c in range(5) if class_counts[c] > 0]
sizes_pie    = [class_counts[c] for c in range(5) if class_counts[c] > 0]
colors_pie   = [class_colors[c]  for c in range(5) if class_counts[c] > 0]
axes[1].pie(sizes_pie, labels=labels_pie, colors=colors_pie,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution')

plt.suptitle(
    f'Samsung Galaxy Watch ECG — Results\n'
    f'BPM: {dominant_bpm:.1f}  |  '
    f'Total beats: {len(y_pred)}  |  '
    f'Mean confidence: {conf.mean()*100:.1f}%  |  '
    f'Diagnosis: {diagnosis}',
    fontsize=10
)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ecg_confidence.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  ✓ Saved: ecg_confidence.png")

print(f"\n{'='*45}")
print(f"  Final Diagnosis: {diagnosis}")
print(f"{'='*45}")